In [1]:
# ============================================================
# C7 — Step 1: Imports
# ============================================================

import pandas as pd
import numpy as np
import os
import time

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    median_absolute_error,
    explained_variance_score
)

print("C7 imports completed successfully!")

C7 imports completed successfully!


In [2]:
# ============================================================
# C7 — Step 2: Load Feature-Engineered Energy & Weather Data
# ============================================================

energy_fe_path = "../../Feature_engineered_datasets/energy_feature_engineered.csv"
weather_fe_path = "../../Feature_engineered_datasets/weather_feature_engineered.csv"

energy_df = pd.read_csv(energy_fe_path)
weather_df = pd.read_csv(weather_fe_path)

print("Energy Dataset Shape:", energy_df.shape)
print("Weather Dataset Shape:", weather_df.shape)

print("\nEnergy Columns:")
print(energy_df.columns.tolist())

print("\nWeather Columns:")
print(weather_df.columns.tolist())

Energy Dataset Shape: (1000, 19)
Weather Dataset Shape: (96429, 18)

Energy Columns:
['Timestamp', 'Temperature', 'Humidity', 'SquareFootage', 'Occupancy', 'HVACUsage', 'LightingUsage', 'RenewableEnergy', 'DayOfWeek', 'Holiday', 'EnergyConsumption', 'Hour', 'Month', 'Weekend', 'HVAC_Status', 'Lighting_Status', 'Energy_Per_Occupant', 'Renewable_Energy_Ratio', 'Energy_Per_SquareFoot']

Weather Columns:
['Formatted Date', 'Summary', 'Precip Type', 'Temperature (C)', 'Apparent Temperature (C)', 'Humidity', 'Wind Speed (km/h)', 'Wind Bearing (degrees)', 'Visibility (km)', 'Pressure (millibars)', 'Daily Summary', 'Year', 'Month', 'Hour', 'DayOfWeek', 'Season', 'Temperature_Difference', 'Wind_Chill_Indicator']


In [3]:
# ============================================================
# C7 — Step 3: Target and Leakage Check
# ============================================================

target = "EnergyConsumption"

# Energy columns that are derived from the target
leakage_columns = [
    "Energy_Per_Occupant",
    "Renewable_Energy_Ratio",
    "Energy_Per_SquareFoot"
]

print("Target:")
print(target)

print("\nPotential Leakage Columns:")
for col in leakage_columns:
    print("-", col)

print("\nLeakage Check:")
for col in leakage_columns:
    print(
        f"{col}:",
        "FOUND" if col in energy_df.columns else "NOT FOUND"
    )

print("\nTarget present:", target in energy_df.columns)

Target:
EnergyConsumption

Potential Leakage Columns:
- Energy_Per_Occupant
- Renewable_Energy_Ratio
- Energy_Per_SquareFoot

Leakage Check:
Energy_Per_Occupant: FOUND
Renewable_Energy_Ratio: FOUND
Energy_Per_SquareFoot: FOUND

Target present: True


In [4]:
# ============================================================
# C7 — Step 4: Prepare Energy Features
# ============================================================

energy_features_c7 = [
    "Temperature",
    "Humidity",
    "SquareFootage",
    "Occupancy",
    "HVACUsage",
    "LightingUsage",
    "RenewableEnergy",
    "DayOfWeek",
    "Holiday",
    "Hour",
    "Month",
    "Weekend",
    "HVAC_Status",
    "Lighting_Status"
]

X_energy_c7 = energy_df[energy_features_c7].copy()
y_energy_c7 = energy_df[target].copy()

print("Energy Feature Matrix Shape:", X_energy_c7.shape)
print("Target Shape:", y_energy_c7.shape)

print("\nEnergy Features:")
print(X_energy_c7.columns.tolist())

print("\nMissing Values:", X_energy_c7.isna().sum().sum())
print("Target Missing Values:", y_energy_c7.isna().sum())

Energy Feature Matrix Shape: (1000, 14)
Target Shape: (1000,)

Energy Features:
['Temperature', 'Humidity', 'SquareFootage', 'Occupancy', 'HVACUsage', 'LightingUsage', 'RenewableEnergy', 'DayOfWeek', 'Holiday', 'Hour', 'Month', 'Weekend', 'HVAC_Status', 'Lighting_Status']

Missing Values: 0
Target Missing Values: 0


In [5]:
# ============================================================
# C7 — Step 5: Encode Energy Features
# ============================================================

categorical_energy_columns = [
    "HVACUsage",
    "LightingUsage",
    "DayOfWeek",
    "Holiday",
    "Weekend"
]

X_energy_c7_encoded = pd.get_dummies(
    X_energy_c7,
    columns=categorical_energy_columns,
    drop_first=True,
    dtype=int
)

print("Encoded Energy Feature Matrix Shape:",
      X_energy_c7_encoded.shape)

print("\nEncoded Columns:")
print(X_energy_c7_encoded.columns.tolist())

print("\nMissing Values:",
      X_energy_c7_encoded.isna().sum().sum())

Encoded Energy Feature Matrix Shape: (1000, 19)

Encoded Columns:
['Temperature', 'Humidity', 'SquareFootage', 'Occupancy', 'RenewableEnergy', 'Hour', 'Month', 'HVAC_Status', 'Lighting_Status', 'HVACUsage_On', 'LightingUsage_On', 'DayOfWeek_Monday', 'DayOfWeek_Saturday', 'DayOfWeek_Sunday', 'DayOfWeek_Thursday', 'DayOfWeek_Tuesday', 'DayOfWeek_Wednesday', 'Holiday_Yes', 'Weekend_Yes']

Missing Values: 0


In [6]:
# ============================================================
# C7 — Step 6: Prepare Weather Context
# ============================================================

weather_context_features_c7 = [
    "Temperature (C)",
    "Apparent Temperature (C)",
    "Humidity",
    "Wind Speed (km/h)",
    "Visibility (km)",
    "Pressure (millibars)"
]

weather_context_c7 = weather_df[
    ["Month", "Hour"] + weather_context_features_c7
].copy()

print("Weather Context Shape:",
      weather_context_c7.shape)

print("\nWeather Context Columns:")
print(weather_context_c7.columns.tolist())

print("\nMissing Values:")
print(weather_context_c7.isna().sum())

Weather Context Shape: (96429, 8)

Weather Context Columns:
['Month', 'Hour', 'Temperature (C)', 'Apparent Temperature (C)', 'Humidity', 'Wind Speed (km/h)', 'Visibility (km)', 'Pressure (millibars)']

Missing Values:
Month                       0
Hour                        0
Temperature (C)             0
Apparent Temperature (C)    0
Humidity                    0
Wind Speed (km/h)           0
Visibility (km)             0
Pressure (millibars)        0
dtype: int64


In [7]:
# ============================================================
# C7 — Step 7: Aggregate Weather Context
# ============================================================

weather_context_avg_c7 = (
    weather_context_c7
    .groupby(["Month", "Hour"])[weather_context_features_c7]
    .mean()
    .reset_index()
)

print("Aggregated Weather Context Shape:",
      weather_context_avg_c7.shape)

print("\nExpected Month-Hour Groups: 288")

print("\nNumber of Month-Hour Groups:",
      len(weather_context_avg_c7))

print("\nFirst 10 Groups:")
print(weather_context_avg_c7.head(10))

print("\nMissing Values:",
      weather_context_avg_c7.isna().sum().sum())

Aggregated Weather Context Shape: (288, 8)

Expected Month-Hour Groups: 288

Number of Month-Hour Groups: 288

First 10 Groups:
   Month  Hour  Temperature (C)  Apparent Temperature (C)  Humidity  \
0      1     0        -0.291105                 -2.963718  0.883607   
1      1     1        -0.410427                 -3.252786  0.881056   
2      1     2        -0.566846                 -3.283399  0.883959   
3      1     3        -0.614614                 -3.371717  0.889795   
4      1     4        -0.772988                 -3.598110  0.886041   
5      1     5        -0.790893                 -3.610101  0.887947   
6      1     6        -0.761518                 -3.637341  0.890528   
7      1     7        -0.457592                 -3.307739  0.884839   
8      1     8         0.391544                 -2.569762  0.864516   
9      1     9         1.311274                 -1.705279  0.841173   

   Wind Speed (km/h)  Visibility (km)  Pressure (millibars)  
0          10.346729        

In [8]:
# ============================================================
# C7 — Step 8: Fuse Weather Context with Energy
# ============================================================

energy_weather_c7 = energy_df.merge(
    weather_context_avg_c7,
    on=["Month", "Hour"],
    how="left",
    suffixes=("", "_Weather")
)

print("Fused Dataset Shape:",
      energy_weather_c7.shape)

print("\nFused Dataset Columns:")
print(energy_weather_c7.columns.tolist())

print("\nMissing Values in Weather Features:")
print(
    energy_weather_c7[
        weather_context_features_c7
    ].isna().sum()
)

print("\nTotal Missing Values:",
      energy_weather_c7.isna().sum().sum())

Fused Dataset Shape: (1000, 25)

Fused Dataset Columns:
['Timestamp', 'Temperature', 'Humidity', 'SquareFootage', 'Occupancy', 'HVACUsage', 'LightingUsage', 'RenewableEnergy', 'DayOfWeek', 'Holiday', 'EnergyConsumption', 'Hour', 'Month', 'Weekend', 'HVAC_Status', 'Lighting_Status', 'Energy_Per_Occupant', 'Renewable_Energy_Ratio', 'Energy_Per_SquareFoot', 'Temperature (C)', 'Apparent Temperature (C)', 'Humidity_Weather', 'Wind Speed (km/h)', 'Visibility (km)', 'Pressure (millibars)']

Missing Values in Weather Features:
Temperature (C)             0
Apparent Temperature (C)    0
Humidity                    0
Wind Speed (km/h)           0
Visibility (km)             0
Pressure (millibars)        0
dtype: int64

Total Missing Values: 0


In [9]:
# ============================================================
# C7 — Step 9: Cross-Domain Interaction Features
# ============================================================

energy_weather_c7["Weather_Energy_Temp_Difference"] = (
    energy_weather_c7["Temperature (C)"]
    - energy_weather_c7["Temperature"]
)

energy_weather_c7["Weather_Energy_Humidity_Difference"] = (
    energy_weather_c7["Humidity_Weather"]
    - energy_weather_c7["Humidity"]
)

energy_weather_c7["Occupancy_Density"] = (
    energy_weather_c7["Occupancy"]
    / (energy_weather_c7["SquareFootage"] + 1e-6)
)

interaction_features_c7 = [
    "Weather_Energy_Temp_Difference",
    "Weather_Energy_Humidity_Difference",
    "Occupancy_Density"
]

print("Created Interaction Features:")
print(interaction_features_c7)

print("\nInteraction Feature Statistics:")
print(
    energy_weather_c7[
        interaction_features_c7
    ].describe()
)

print("\nMissing Values:")
print(
    energy_weather_c7[
        interaction_features_c7
    ].isna().sum()
)

Created Interaction Features:
['Weather_Energy_Temp_Difference', 'Weather_Energy_Humidity_Difference', 'Occupancy_Density']

Interaction Feature Statistics:
       Weather_Energy_Temp_Difference  Weather_Energy_Humidity_Difference  \
count                     1000.000000                         1000.000000   
mean                       -23.819567                          -44.554390   
std                          3.280384                            8.517585   
min                        -30.718398                          -59.146489   
25%                        -26.274141                          -51.547936   
50%                        -23.636125                          -45.130802   
75%                        -21.494904                          -37.541926   
max                        -14.747672                          -29.125447   

       Occupancy_Density  
count        1000.000000  
mean            0.003171  
std             0.002108  
min             0.000000  
25%           

In [10]:
# ============================================================
# C7 — Step 10: Build Leakage-Free Fused Feature Matrix
# ============================================================

energy_features_for_fusion_c7 = [
    "Temperature",
    "Humidity",
    "SquareFootage",
    "Occupancy",
    "HVACUsage",
    "LightingUsage",
    "RenewableEnergy",
    "DayOfWeek",
    "Holiday",
    "Hour",
    "Month",
    "Weekend",
    "HVAC_Status",
    "Lighting_Status"
]

weather_features_for_fusion_c7 = [
    "Temperature (C)",
    "Apparent Temperature (C)",
    "Humidity_Weather",
    "Wind Speed (km/h)",
    "Visibility (km)",
    "Pressure (millibars)"
]

X_fused_c7 = energy_weather_c7[
    energy_features_for_fusion_c7
    + weather_features_for_fusion_c7
    + interaction_features_c7
].copy()

y_fused_c7 = energy_weather_c7[target].copy()

print("Initial Fused Feature Matrix Shape:",
      X_fused_c7.shape)

print("\nTarget Shape:",
      y_fused_c7.shape)

print("\nFeatures Before Encoding:")
print(X_fused_c7.columns.tolist())

print("\nMissing Values:",
      X_fused_c7.isna().sum().sum())

print("\nTarget Missing Values:",
      y_fused_c7.isna().sum())

Initial Fused Feature Matrix Shape: (1000, 23)

Target Shape: (1000,)

Features Before Encoding:
['Temperature', 'Humidity', 'SquareFootage', 'Occupancy', 'HVACUsage', 'LightingUsage', 'RenewableEnergy', 'DayOfWeek', 'Holiday', 'Hour', 'Month', 'Weekend', 'HVAC_Status', 'Lighting_Status', 'Temperature (C)', 'Apparent Temperature (C)', 'Humidity_Weather', 'Wind Speed (km/h)', 'Visibility (km)', 'Pressure (millibars)', 'Weather_Energy_Temp_Difference', 'Weather_Energy_Humidity_Difference', 'Occupancy_Density']

Missing Values: 0

Target Missing Values: 0


In [11]:
# ============================================================
# C7 — Step 11: One-Hot Encode Fused Features
# ============================================================

categorical_fused_columns_c7 = [
    "HVACUsage",
    "LightingUsage",
    "DayOfWeek",
    "Holiday",
    "Weekend"
]

X_fused_c7_encoded = pd.get_dummies(
    X_fused_c7,
    columns=categorical_fused_columns_c7,
    drop_first=True,
    dtype=int
)

print("Encoded Fused Feature Matrix Shape:",
      X_fused_c7_encoded.shape)

print("\nEncoded Features:")
print(X_fused_c7_encoded.columns.tolist())

print("\nMissing Values:",
      X_fused_c7_encoded.isna().sum().sum())

print("\nData Types:")
print(X_fused_c7_encoded.dtypes.value_counts())

Encoded Fused Feature Matrix Shape: (1000, 28)

Encoded Features:
['Temperature', 'Humidity', 'SquareFootage', 'Occupancy', 'RenewableEnergy', 'Hour', 'Month', 'HVAC_Status', 'Lighting_Status', 'Temperature (C)', 'Apparent Temperature (C)', 'Humidity_Weather', 'Wind Speed (km/h)', 'Visibility (km)', 'Pressure (millibars)', 'Weather_Energy_Temp_Difference', 'Weather_Energy_Humidity_Difference', 'Occupancy_Density', 'HVACUsage_On', 'LightingUsage_On', 'DayOfWeek_Monday', 'DayOfWeek_Saturday', 'DayOfWeek_Sunday', 'DayOfWeek_Thursday', 'DayOfWeek_Tuesday', 'DayOfWeek_Wednesday', 'Holiday_Yes', 'Weekend_Yes']

Missing Values: 0

Data Types:
int64      15
float64    13
Name: count, dtype: int64


In [12]:
# ============================================================
# C7 — Step 12: Train-Test Split
# ============================================================

X_train_c7, X_test_c7, y_train_c7, y_test_c7 = train_test_split(
    X_fused_c7_encoded,
    y_fused_c7,
    test_size=0.20,
    random_state=42
)

print("Training Feature Shape:",
      X_train_c7.shape)

print("Testing Feature Shape:",
      X_test_c7.shape)

print("\nTraining Target Shape:",
      y_train_c7.shape)

print("Testing Target Shape:",
      y_test_c7.shape)

print("\nTraining Samples:",
      len(X_train_c7))

print("Testing Samples:",
      len(X_test_c7))

Training Feature Shape: (800, 28)
Testing Feature Shape: (200, 28)

Training Target Shape: (800,)
Testing Target Shape: (200,)

Training Samples: 800
Testing Samples: 200


In [13]:
# ============================================================
# C7 — Step 13: Fused Linear Regression
# ============================================================

c7_lr = LinearRegression()

start_time = time.time()

c7_lr.fit(X_train_c7, y_train_c7)

y_pred_c7_lr = c7_lr.predict(X_test_c7)

c7_lr_time = time.time() - start_time

# Metrics
c7_lr_mae = mean_absolute_error(y_test_c7, y_pred_c7_lr)
c7_lr_mse = mean_squared_error(y_test_c7, y_pred_c7_lr)
c7_lr_rmse = np.sqrt(c7_lr_mse)
c7_lr_r2 = r2_score(y_test_c7, y_pred_c7_lr)
c7_lr_medae = median_absolute_error(y_test_c7, y_pred_c7_lr)
c7_lr_ev = explained_variance_score(y_test_c7, y_pred_c7_lr)

# sMAPE
c7_lr_smape = (
    100 * np.mean(
        2 * np.abs(y_test_c7 - y_pred_c7_lr)
        / (
            np.abs(y_test_c7)
            + np.abs(y_pred_c7_lr)
            + 1e-8
        )
    )
)

# NRMSE using mean target
c7_lr_nrmse = (
    c7_lr_rmse / np.mean(y_test_c7)
) * 100

print("C7 Fused Linear Regression Results")
print("-" * 45)

print(f"MAE:               {c7_lr_mae:.4f}")
print(f"MSE:               {c7_lr_mse:.4f}")
print(f"RMSE:              {c7_lr_rmse:.4f}")
print(f"R²:                {c7_lr_r2:.4f}")
print(f"sMAPE:             {c7_lr_smape:.4f}%")
print(f"NRMSE:             {c7_lr_nrmse:.4f}%")
print(f"Median Absolute Error: {c7_lr_medae:.4f}")
print(f"Explained Variance:    {c7_lr_ev:.4f}")
print(f"Training Time:         {c7_lr_time:.4f} seconds")

C7 Fused Linear Regression Results
---------------------------------------------
MAE:               4.2218
MSE:               27.9129
RMSE:              5.2833
R²:                0.5738
sMAPE:             5.5399%
NRMSE:             6.8541%
Median Absolute Error: 3.6630
Explained Variance:    0.5739
Training Time:         0.0399 seconds


In [14]:
# ============================================================
# C7 — Step 14: Fused Random Forest
# ============================================================

c7_rf = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

start_time = time.time()

c7_rf.fit(X_train_c7, y_train_c7)

y_pred_c7_rf = c7_rf.predict(X_test_c7)

c7_rf_time = time.time() - start_time

# Metrics
c7_rf_mae = mean_absolute_error(y_test_c7, y_pred_c7_rf)
c7_rf_mse = mean_squared_error(y_test_c7, y_pred_c7_rf)
c7_rf_rmse = np.sqrt(c7_rf_mse)
c7_rf_r2 = r2_score(y_test_c7, y_pred_c7_rf)
c7_rf_medae = median_absolute_error(y_test_c7, y_pred_c7_rf)
c7_rf_ev = explained_variance_score(y_test_c7, y_pred_c7_rf)

# sMAPE
c7_rf_smape = (
    100 * np.mean(
        2 * np.abs(y_test_c7 - y_pred_c7_rf)
        / (
            np.abs(y_test_c7)
            + np.abs(y_pred_c7_rf)
            + 1e-8
        )
    )
)

# NRMSE
c7_rf_nrmse = (
    c7_rf_rmse / np.mean(y_test_c7)
) * 100

print("C7 Fused Random Forest Results")
print("-" * 45)

print(f"MAE:                  {c7_rf_mae:.4f}")
print(f"MSE:                  {c7_rf_mse:.4f}")
print(f"RMSE:                 {c7_rf_rmse:.4f}")
print(f"R²:                   {c7_rf_r2:.4f}")
print(f"sMAPE:                {c7_rf_smape:.4f}%")
print(f"NRMSE:                {c7_rf_nrmse:.4f}%")
print(f"Median Absolute Error:{c7_rf_medae:.4f}")
print(f"Explained Variance:   {c7_rf_ev:.4f}")
print(f"Training Time:        {c7_rf_time:.4f} seconds")

C7 Fused Random Forest Results
---------------------------------------------
MAE:                  4.4006
MSE:                  29.9817
RMSE:                 5.4756
R²:                   0.5423
sMAPE:                5.7545%
NRMSE:                7.1036%
Median Absolute Error:4.0825
Explained Variance:   0.5423
Training Time:        0.2699 seconds


In [15]:
# ============================================================
# C7 — Step 15: Validation-Based LR + RF Hybrid
# ============================================================

from sklearn.model_selection import train_test_split

# ------------------------------------------------------------
# 1. Split the original training data into train + validation
# ------------------------------------------------------------

X_train_inner, X_val_c7, y_train_inner, y_val_c7 = train_test_split(
    X_train_c7,
    y_train_c7,
    test_size=0.20,
    random_state=42
)

print("Inner Training Shape:", X_train_inner.shape)
print("Validation Shape:", X_val_c7.shape)


# ------------------------------------------------------------
# 2. Train LR on inner training data
# ------------------------------------------------------------

lr_inner_c7 = LinearRegression()

lr_inner_c7.fit(
    X_train_inner,
    y_train_inner
)

y_val_lr_c7 = lr_inner_c7.predict(X_val_c7)


# ------------------------------------------------------------
# 3. Train RF on inner training data
# ------------------------------------------------------------

rf_inner_c7 = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

rf_inner_c7.fit(
    X_train_inner,
    y_train_inner
)

y_val_rf_c7 = rf_inner_c7.predict(X_val_c7)


# ------------------------------------------------------------
# 4. Search for the best blending weight
# ------------------------------------------------------------

weights = np.arange(0.0, 1.01, 0.05)

weight_results = []

for w in weights:

    y_val_hybrid = (
        w * y_val_lr_c7
        + (1 - w) * y_val_rf_c7
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_val_c7,
            y_val_hybrid
        )
    )

    weight_results.append({
        "LR_Weight": w,
        "RF_Weight": 1 - w,
        "Validation_RMSE": rmse
    })


weight_results_df = pd.DataFrame(weight_results)

best_row = weight_results_df.loc[
    weight_results_df["Validation_RMSE"].idxmin()
]

best_lr_weight = best_row["LR_Weight"]
best_rf_weight = best_row["RF_Weight"]

print("\nBlending Weight Search:")
print(weight_results_df)

print("\nBest Validation Weight:")
print(f"LR Weight: {best_lr_weight:.2f}")
print(f"RF Weight: {best_rf_weight:.2f}")
print(
    f"Validation RMSE: "
    f"{best_row['Validation_RMSE']:.4f}"
)

Inner Training Shape: (640, 28)
Validation Shape: (160, 28)

Blending Weight Search:
    LR_Weight  RF_Weight  Validation_RMSE
0        0.00       1.00         5.656461
1        0.05       0.95         5.624435
2        0.10       0.90         5.594940
3        0.15       0.85         5.568018
4        0.20       0.80         5.543706
5        0.25       0.75         5.522038
6        0.30       0.70         5.503046
7        0.35       0.65         5.486757
8        0.40       0.60         5.473196
9        0.45       0.55         5.462382
10       0.50       0.50         5.454333
11       0.55       0.45         5.449060
12       0.60       0.40         5.446572
13       0.65       0.35         5.446872
14       0.70       0.30         5.449960
15       0.75       0.25         5.455831
16       0.80       0.20         5.464477
17       0.85       0.15         5.475883
18       0.90       0.10         5.490033
19       0.95       0.05         5.506906
20       1.00       0.00         

In [16]:
# ============================================================
# C7 — Step 16: Final LR + RF Hybrid
# ============================================================

# ------------------------------------------------------------
# 1. Train Linear Regression on full training data
# ------------------------------------------------------------

c7_hybrid_lr = LinearRegression()

c7_hybrid_lr.fit(
    X_train_c7,
    y_train_c7
)

y_test_lr_hybrid = c7_hybrid_lr.predict(
    X_test_c7
)


# ------------------------------------------------------------
# 2. Train Random Forest on full training data
# ------------------------------------------------------------

c7_hybrid_rf = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

start_time = time.time()

c7_hybrid_rf.fit(
    X_train_c7,
    y_train_c7
)

rf_training_time = time.time() - start_time

y_test_rf_hybrid = c7_hybrid_rf.predict(
    X_test_c7
)


# ------------------------------------------------------------
# 3. Generate final hybrid prediction
# ------------------------------------------------------------

y_pred_c7_hybrid = (
    best_lr_weight * y_test_lr_hybrid
    + best_rf_weight * y_test_rf_hybrid
)


# ------------------------------------------------------------
# 4. Evaluate hybrid
# ------------------------------------------------------------

c7_hybrid_mae = mean_absolute_error(
    y_test_c7,
    y_pred_c7_hybrid
)

c7_hybrid_mse = mean_squared_error(
    y_test_c7,
    y_pred_c7_hybrid
)

c7_hybrid_rmse = np.sqrt(
    c7_hybrid_mse
)

c7_hybrid_r2 = r2_score(
    y_test_c7,
    y_pred_c7_hybrid
)

c7_hybrid_medae = median_absolute_error(
    y_test_c7,
    y_pred_c7_hybrid
)

c7_hybrid_ev = explained_variance_score(
    y_test_c7,
    y_pred_c7_hybrid
)


# ------------------------------------------------------------
# 5. sMAPE
# ------------------------------------------------------------

c7_hybrid_smape = (
    100 * np.mean(
        2 * np.abs(
            y_test_c7 - y_pred_c7_hybrid
        )
        /
        (
            np.abs(y_test_c7)
            + np.abs(y_pred_c7_hybrid)
            + 1e-8
        )
    )
)


# ------------------------------------------------------------
# 6. NRMSE
# ------------------------------------------------------------

c7_hybrid_nrmse = (
    c7_hybrid_rmse
    / np.mean(y_test_c7)
) * 100


# ------------------------------------------------------------
# 7. Print results
# ------------------------------------------------------------

print("C7 Hybrid — Linear Regression + Random Forest")
print("=" * 55)

print(f"LR Weight:             {best_lr_weight:.2f}")
print(f"RF Weight:             {best_rf_weight:.2f}")

print("\nTest Set Performance")
print("-" * 55)

print(f"MAE:                   {c7_hybrid_mae:.4f}")
print(f"MSE:                   {c7_hybrid_mse:.4f}")
print(f"RMSE:                  {c7_hybrid_rmse:.4f}")
print(f"R²:                    {c7_hybrid_r2:.4f}")
print(f"sMAPE:                 {c7_hybrid_smape:.4f}%")
print(f"NRMSE:                 {c7_hybrid_nrmse:.4f}%")
print(f"Median Absolute Error: {c7_hybrid_medae:.4f}")
print(f"Explained Variance:    {c7_hybrid_ev:.4f}")

C7 Hybrid — Linear Regression + Random Forest
LR Weight:             0.60
RF Weight:             0.40

Test Set Performance
-------------------------------------------------------
MAE:                   4.2307
MSE:                   27.8762
RMSE:                  5.2798
R²:                    0.5744
sMAPE:                 5.5428%
NRMSE:                 6.8496%
Median Absolute Error: 3.8575
Explained Variance:    0.5744
